# Tutorial 1: PyTerrier Indexing

In diesem Hands On Tutorial werden wir uns die LongEval Retrieval Test Collection anschauen.


Hierzu verwenden wir [PyTerrier](https://pyterrier.readthedocs.io/en/latest/index.html) und [Pandas](https://pandas.pydata.org/). PyTerrier ist ein praktisches Retrieval Toolkit, mit dem schnell und einfach Retrieval-Systeme erstellt erstellt und getestet werden können. Dabei liegt der Fokus auf experimentellen Evaluationen, die Systeme skalieren also nicht auf Produktionsniveau. Dafür werden viele praktische Tools bereitgestellt, um Evaluationen schnell zu ermöglichen.

Weitere Informationen zu dem Datensatz und dem shared task finden sich zum Beispiel unter:
- [LongEval Overview](https://clef-longeval.github.io/2025/data/)

In diesem ersten Hands-on Notebook werden wir uns die Topics, Dokumente und Qrels der Test Kollektion anschauen und dabei den Umgang mit PyTerrier lernen.

Als erstes müssen wir den LongEval Datensatz herunterladen. Praktischer weise haben wir eine integration für das ir_datasets toolkit erstellt die euch den Einsatz erleichtert.

Zunächst müssen wir wieder PyTerrier installieren. Hierzu müssen wir nur diese Zelle ausführen:

In [ ]:
!pip install -q python-terrier ir_datasets_longeval

Bevor wir PyTerrier benutzen können müssen wir es importieren und initialisieren.

In [ ]:
import pyterrier as pt

Als erstes laden wir den datensatz herunter. Dazu können wir die `load` Funktion in `ir_datasets_longeval` verweden. Fur diesen test laden wir nur den *train* sliche des `2024-11` snapshots. Insgesammt müsen ca. 2 GB heruntergeladen werden, es kann also einen kleinen Moment dauern. 

In [ ]:
from ir_datasets_longeval import load

In [ ]:
dataset = load("longeval-sci/2024-11/train")

Um den Datensatz zu durchsuchen, müssen wir ihn indexieren. PyTerrier bietet die Möglichkeit Datensätze in verschiedenen Formaten zu indexieren. Der kleine LongEval Datensatz liegt im TREC Format (XML) vor. Schau dir die Dateien am besten im Dateibrowser an, um einen Überblick zu bekommen welche Informationen zur Verfügung stehen.

Um TREC Kollektionen zu indexieren, nutzen wir den `TRECCollectionIndexer`. Die Details  und ein Überblick über andere Indexer sind in der [PyTerrier Dokumentation](https://pyterrier.readthedocs.io/en/latest/terrier-indexing.html#treccollectionindexer) zu finden.

In [ ]:
indexer = pt.IterDictIndexer(
    index_path="./index", # path to the index
    overwrite=True, 
    meta={"docno": 100, "text": 20480}  # add the docno and text fields to the index metadata
)

In [ ]:
# you can do some custom document processing here
def document_generator():
    for doc in dataset.docs_iter():
        yield {
            "docno": doc.doc_id,
            "text": doc.default_text()
        }

docs = document_generator()

In [ ]:
indexer.index(docs)

Die Indexierung dauert ca. 10 min. Du solltest nun einen Ordner mit dem Namen `index` haben in dem der fertige Index liegt. Mithilfe der folgenden Funktion können wir den Index aus dem Ordner laden und im Anschluss durchsuchen.

In [ ]:
index = pt.IndexFactory.of("./index")

Zur Überprüfung das alles funktioniert hat lassen wir uns noch die Anzahl der Dokumente im Index anzeigen. Es sollten 2.014.265 Dokumente sein.

In [ ]:
print(index.getCollectionStatistics().toString())